# Card Fraud Detection: Data Exploration

In this notebook, we will perform some initial analysis on the Card Fraud Dataset.

The original dataset is publicly available on Kaggle: [https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

In [ ]:
import pandas as pd
import os

project_dir = os.getenv("PROJECT_DIR")
env = os.getenv("CONDA_DEFAULT_ENV")
dataset_csv = os.getenv("DATA")

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
print(dataset_csv)

In [ ]:
df = pd.read_csv(dataset_csv)

In [ ]:
df.head()

Checking Class Distribution and (any) Missing Value

In [ ]:
print("data shape is", df.shape)

count_nan = df.isna().sum().sum()
print("Number of Missing Values (NaN) \n", count_nan)
# data is clean without nan

# Distribution of classes
fraud_ratio = (df["Class"].value_counts()[1]) / len(df) * 100
non_fraud_ratio = (df["Class"].value_counts()[0]) / len(df) * 100


print("Fraud/Non Fraud Transactions:  {:.2f}/{:.2f}%".format(fraud_ratio, non_fraud_ratio))

In [ ]:
import seaborn as sns

from matplotlib import pyplot as plt

In [ ]:
def plot_samples(df, title="Class Distributions"):
    colors = ["orange", "green"]  # ["#0101DF", "#DF0101"]
    print("data size is {}".format(df.shape))
    ax = sns.countplot(x="Class", data=df, palette=colors)
    for container in ax.containers:
        ax.bar_label(container)
    plt.title(f"{title} \n (0: No Fraud || 1: Fraud)", fontsize=14)

    plt.show()


plot_samples(df)

### Data Balancing

In [ ]:
from imblearn.under_sampling import NearMiss
from imblearn.over_sampling import SMOTE

In [ ]:
import numpy as np
from sklearn.utils import check_random_state

SEED = 12345

# The NumPy Generator will be used throughout the whole experiment
# rng = np.random.default_rng(SEED)
np.random.seed(SEED)
rng = check_random_state(SEED)

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df[df.columns[df.columns != "Class"]], df["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=rng)

##### Feature Scaling

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

# (Selected) Feature Scaling
preprocessing = ColumnTransformer(
    [
        ("scaler", RobustScaler(), ["Time", "Amount"]),
    ],
    remainder="passthrough",
)

In [ ]:
X_train_scaled = preprocessing.fit_transform(X_train)
X_test_scaled = preprocessing.transform(X_test)

#### Under sampling: Near Miss

In [ ]:
# Under Sampling Strategy
nm = NearMiss(sampling_strategy="majority", version=3, n_neighbors_ver3=5)
X_train_scaled_sampled, y_train_sampled = nm.fit_resample(X_train_scaled, y_train)

In [ ]:
X_train_scaled_sampled.shape, y_train_sampled.shape

In [ ]:
X_train_scaled.shape, y_train.shape

In [ ]:
y_train[y_train == 1].shape

In [ ]:
y_train_sampled[y_train_sampled == 0].shape == y_train_sampled[y_train_sampled == 1].shape

In [ ]:
df_sampled_nm = pd.DataFrame(X_train_scaled_sampled, columns=X_train.columns)
df_sampled_nm["Class"] = y_train_sampled

df_sampled_nm.shape

In [ ]:
plot_samples(df_sampled_nm, title="Class Distribution after Under Sampling (Near Miss)")

#### Over sampling: SMOTE

In [ ]:
# Over Sampling Strategies
smote = SMOTE(sampling_strategy="minority", random_state=rng)
X_train_scaled_sampled, y_train_sampled = smote.fit_resample(X_train_scaled, y_train)

In [ ]:
X_train_scaled_sampled.shape, y_train_sampled.shape

In [ ]:
X_train_scaled.shape, y_train.shape

In [ ]:
y_train[y_train == 1].shape

In [ ]:
y_train_sampled[y_train_sampled == 0].shape == y_train_sampled[y_train_sampled == 1].shape

In [ ]:
df_sampled_smote = pd.DataFrame(X_train_scaled_sampled, columns=X_train.columns)
df_sampled_smote["Class"] = y_train_sampled

df_sampled_smote.shape

In [ ]:
plot_samples(df_sampled_smote, title="Class Distribution after Over Sampling (SMOTE)")

### Feature Importance

In [ ]:
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

In [ ]:
best_feature_count = 5

KBestScores = SelectKBest(score_func=f_classif, k=best_feature_count).fit(X_train, y_train)

# summarize scores
np.set_printoptions(precision=3)

print("scores_:", KBestScores.scores_)
print("selected index:", KBestScores.get_support(True))


feat_names = df.columns.values[KBestScores.get_support(True)]
indices = np.argsort(KBestScores.scores_)[::-1]
print("selected features {}".format(feat_names))

In [ ]:
plt.figure()
plt.title("SelectKBestScores Top 5 features", fontsize=18)
plt.xlabel("Feature Name", fontsize=16)
plt.ylabel("Feature Score", fontsize=16)
plt.bar(feat_names, KBestScores.scores_[indices[range(best_feature_count)]], color="r", align="center")
plt.show()